# 6. 모델 성능 개선과 검증

- 목표: 로지스틱 회귀, 전처리, 스케일링, 규제, 교차 검증, 파이프라인을 묶어 성능 개선 흐름을 익힙니다.
- 흐름: 마지막에는 타이타닉 분류 베이스라인으로 내용을 정리합니다.


## 1. 로지스틱 회귀 (Logistic Regression)

### 1.1 개념
- **선형 회귀**는 연속값 예측 → 회귀 문제에 적합  
- **로지스틱 회귀**는 범주형 값 예측 → 분류 문제에 적합  
- 예: 스팸메일(스팸/정상), 시험 결과(합격/불합격)


### 1.2 시그모이드 함수 (Sigmoid Function)
- 로지스틱 회귀의 핵심: 입력값을 **0~1 사이 확률**로 변환  

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

- z = \( w x + b \)  


<img src="image/sigmoid.jpg" alt="sigmoid" width="400">

이미지 출처 : https://en.wikipedia.org/wiki/Sigmoid_function


In [ ]:
import numpy as np  # 배열과 수치 연산 라이브러리입니다.
import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

x = np.linspace(-10, 10, 100)  # 일정 간격의 숫자 배열을 만듭니다.
y = sigmoid(x)

plt.plot(x, y)  # 선 그래프를 그립니다.
plt.title("Sigmoid Function")  # 그래프 제목을 설정합니다.
plt.xlabel("z")  # x축 이름을 설정합니다.
plt.ylabel("σ(z)")  # y축 이름을 설정합니다.
plt.grid()  # 그래프 격자를 표시합니다.
plt.show()  # 그래프를 화면에 출력합니다.

### 1.3 결정 경계 (Decision Boundary)
- 시그모이드 함수 결과 ≥ 0.5 → 1 (True)  
- 시그모이드 함수 결과 < 0.5 → 0 (False)  

즉, 확률을 기준으로 분류하는 것.


### 1.4 손실 함수 (Log Loss)
- 회귀에서는 MSE 사용  
- 분류에서는 **Log Loss (Cross-Entropy Loss)** 사용  


$L = -\frac{1}{n} \sum_{i=1}^n \Big[y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)\Big]$


<img src ="image/logx.jpg" width="500">


### 1.5 Numpy로 구현하기 (이진 분류)


In [ ]:
import numpy as np  # 배열과 수치 연산 라이브러리입니다.

# 데이터: 공부 시간(x) → 합격 여부(y: 0 or 1)
X = np.array([1, 2, 3, 4, 5])  # 넘파이 배열을 만듭니다.
y = np.array([0, 0, 0, 1, 1])  # 넘파이 배열을 만듭니다.

# 파라미터 초기화
w = 0.0
b = 0.0
lr = 0.1
epochs = 1000

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# 경사 하강법
for _ in range(epochs):
    z = w*X + b
    y_pred = sigmoid(z)
    
    # 오차
    error = y_pred - y
    
    # 파라미터 업데이트
    dw = np.dot(error, X) / len(X)  # 두 배열의 내적을 계산합니다.
    db = np.sum(error) / len(X)
    
    w -= lr * dw
    b -= lr * db

print("학습된 w:", w)  # 문자열을 정수로 바꿉니다.
print("학습된 b:", b)  # 문자열을 정수로 바꿉니다.

# 예측
test = np.array([2.5, 3.5, 5])  # 넘파이 배열을 만듭니다.
pred = sigmoid(w*test + b)
print("예측 확률:", pred)  # 문자열을 정수로 바꿉니다.
print("분류 결과:", (pred >= 0.5).astype(int))  # 자료형을 변환합니다.


### 1.6 scikit-learn으로 로지스틱 회귀 구현


In [ ]:
import numpy as np  # 배열과 수치 연산 라이브러리입니다.
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 분류 모델입니다.

X = np.array([1, 2, 3, 4, 5])  # 넘파이 배열을 만듭니다.
y = np.array([0, 0, 0, 1, 1])  # 넘파이 배열을 만듭니다.

X = X.reshape(-1, 1)  # 입력을 2D로 변환
model = LogisticRegression()  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.
model.fit(X, y)  # 데이터로 모델이나 변환 기준을 학습합니다.

print("예측 확률:", model.predict_proba([[3], [4], [5]]))  # 클래스별 예측 확률을 계산합니다.
print("분류 결과:", model.predict([[3], [4], [5]]))  # 학습한 모델로 새 값을 예측합니다.

### 1.7 다중 분류 (Multiclass Classification)
- 로지스틱 회귀는 기본적으로 이진 분류에 사용  
- **OvR(One vs Rest)** 방식으로 다중 분류 확장 가능  
  - 예: 숫자(0~9) 손글씨 분류  



### ✅ 체크포인트
- 로지스틱 회귀는 분류 문제에서 사용되는 지도학습 알고리즘이다.  
- 시그모이드 함수를 사용해 확률(0~1)로 해석할 수 있다.  
- 손실 함수는 Log Loss (교차 엔트로피)를 사용한다.  
- `scikit-learn`으로 손쉽게 이진/다중 분류 문제를 해결할 수 있다.  


## 2. Scikit-learn 모델 사용법 요약

#### 1) 분류(Classification) 예제 — 로지스틱 회귀


In [ ]:
# 분류(Classification) 기본 흐름:
# 1) 더미 데이터 생성 (make_classification)
# 2) 학습/검증 데이터 분리 (train_test_split)
# 3) 모델 생성 및 학습 (LogisticRegression.fit)
# 4) 예측 (predict, predict_proba)
# 5) 성능 평가 (정확도, F1, ROC-AUC, 혼동행렬)``

In [ ]:
from sklearn.datasets import make_classification  # 사이킷런 도구를 불러옵니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 분류 모델입니다.
from sklearn.metrics import (  # 사이킷런 도구를 불러옵니다.
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# 1) 더미 데이터 생성
X, y = make_classification(
    n_samples=1000, # - n_samples: 샘플 수
    n_features=10, # - n_features: 특징 수
    n_informative=5, # - n_informative: 실제로 유용한 특징 수
    n_redundant=2, # - n_redundant: 중복 특징 수
    random_state=42 # - random_state: 재현성
)

# 2) 학습/검증 데이터 분리
# - stratify=y: 분류에서는 클래스 비율을 유지하도록 권장
X_train, X_test, y_train, y_test = train_test_split(  # 데이터를 훈련/평가용으로 나눕니다.
    X, y, test_size=0.2,
    random_state=42, stratify=y
)

# 3) 모델 생성 및 학습
# - max_iter: 수렴 보장을 위해 여유 있게 설정
# - n_jobs: 가능한 경우 병렬 처리 (LogisticRegression 일부 solver에서만 사용)
clf = LogisticRegression(max_iter=1000)  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.
clf.fit(X_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.

# 4) 예측
y_pred = clf.predict(X_test)                    # 라벨 예측(0/1)
y_prob = clf.predict_proba(X_test)[:, 1]        # 양성(1) 클래스 확률

# 5) 성능 평가
acc  = accuracy_score(y_test, y_pred)           # 전체 정확도
prec = precision_score(y_test, y_pred)          # 정밀도(양성 예측의 정확성)
rec  = recall_score(y_test, y_pred)             # 재현율(양성 포착률)
f1   = f1_score(y_test, y_pred)                 # F1(정밀/재현 조화평균)
auc  = roc_auc_score(y_test, y_prob)            # ROC-AUC(임계값 독립 분리도)

cm   = confusion_matrix(y_test, y_pred)         # 혼동행렬
rep  = classification_report(y_test, y_pred)    # 클래스별 정밀/재현/F1 상세

print("[Classification] Logistic Regression")  # 문자열을 정수로 바꿉니다.
print(f"Accuracy  : {acc:.4f}")  # 문자열을 정수로 바꿉니다.
print(f"Precision : {prec:.4f}")  # 문자열을 정수로 바꿉니다.
print(f"Recall    : {rec:.4f}")  # 문자열을 정수로 바꿉니다.
print(f"F1        : {f1:.4f}")  # 문자열을 정수로 바꿉니다.
print(f"ROC-AUC   : {auc:.4f}")  # 문자열을 정수로 바꿉니다.
print("Confusion Matrix:\n", cm)  # 문자열을 정수로 바꿉니다.
print("Classification Report:\n", rep)  # 문자열을 정수로 바꿉니다.


### 기본 지도 학습 알고리즘들 – 실습 문제

### 문제 1. 단순 선형 회귀 (Numpy 구현)
X = [1, 2, 3, 4, 5], y = [2, 4, 6, 8, 10] 데이터를 이용해  
경사 하강법으로 선형 회귀를 학습하고, w와 b를 출력하세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

import numpy as np  # 배열 계산을 위한 Numpy입니다.

X = np.array([1, 2, 3, 4, 5])  # 모델에 넣을 입력 데이터입니다.
y = np.array([2, 4, 6, 8, 10])  # 정답 값 또는 두 번째 배열을 준비합니다.

w, b = 0.0, 0.0  # 모델이 학습할 기울기입니다.
lr = 0.01  # 학습률입니다. 한 번에 이동할 크기입니다.
epochs = 1000  # 전체 학습 반복 횟수입니다.

for _ in range(epochs):  # 값을 하나씩 꺼내 반복합니다.
    y_pred = w*X + b  # 모델 예측 결과를 저장합니다.
    error = y_pred - y  # 예측값과 정답의 차이입니다.
    
    dw = (2/len(X)) * np.dot(error, X)  # 기울기 w를 얼마나 바꿀지 계산한 값입니다.
    db = (2/len(X)) * np.sum(error)  # 절편 b를 얼마나 바꿀지 계산한 값입니다.
    
    w -= lr * dw  # 가중치를 손실 감소 방향으로 갱신합니다.
    b -= lr * db  # 절편을 손실 감소 방향으로 갱신합니다.

print("w:", w, "b:", b)  # 결과를 화면에 출력합니다.

```
</details>

In [ ]:
# 여기에 작성하세요

### 문제 2. 선형 회귀 (scikit-learn)
사이킷런의 `LinearRegression`을 사용해, 공부 시간 X=[1,2,3,4,5]과 점수 y=[2,4,6,8,10] 데이터를 학습하고, 6시간 공부했을 때 점수를 예측하세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.linear_model import LinearRegression  # 선형 회귀 모델입니다.
import numpy as np  # 배열 계산을 위한 Numpy입니다.

X = np.array([1, 2, 3, 4, 5]).reshape(-1, 1)  # 모델에 넣을 입력 데이터입니다.
y = np.array([2, 4, 6, 8, 10])  # 정답 값 또는 두 번째 배열을 준비합니다.

model = LinearRegression()  # 선형 회귀 모델입니다.
model.fit(X, y)  # 데이터로 모델이나 변환 기준을 학습합니다.

print("회귀 계수:", model.coef_)  # 결과를 화면에 출력합니다.
print("절편:", model.intercept_)  # 결과를 화면에 출력합니다.
print("6시간 예측:", model.predict([[6]]))  # 학습한 모델로 새 값을 예측합니다.

```
</details>

In [ ]:
# 여기에 작성하세요
X = np.array([1, 2, 3, 4, 5]).reshape(-1, 1)  # 넘파이 배열을 만듭니다.
y = np.array([2, 4, 6, 8, 10])  # 넘파이 배열을 만듭니다.

### 문제 3. 로지스틱 회귀 (Numpy 구현)
X = [1, 2, 3, 4, 5], y = [0, 0, 0, 1, 1] 데이터에서  
로지스틱 회귀를 경사 하강법으로 학습한 후, X=3, 4, 5에 대한 확률과 분류 결과를 출력하세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

import numpy as np  # 배열 계산을 위한 Numpy입니다.

X = np.array([1, 2, 3, 4, 5])  # 모델에 넣을 입력 데이터입니다.
y = np.array([0, 0, 0, 1, 1])  # 정답 값 또는 두 번째 배열을 준비합니다.

w, b = 0.0, 0.0  # 모델이 학습할 기울기입니다.
lr = 0.1  # 학습률입니다. 한 번에 이동할 크기입니다.
epochs = 1000  # 전체 학습 반복 횟수입니다.

def sigmoid(z):  # 함수를 정의합니다.
    return 1 / (1 + np.exp(-z))  # 계산한 값을 함수 밖으로 돌려줍니다.

for _ in range(epochs):  # 값을 하나씩 꺼내 반복합니다.
    z = w*X + b  # 가중합으로 만든 로짓 값입니다.
    y_pred = sigmoid(z)  # 모델 예측 결과를 저장합니다.
    error = y_pred - y  # 예측값과 정답의 차이입니다.
    
    dw = np.dot(error, X) / len(X)  # 기울기 w를 얼마나 바꿀지 계산한 값입니다.
    db = np.sum(error) / len(X)  # 절편 b를 얼마나 바꿀지 계산한 값입니다.
    
    w -= lr * dw  # 가중치를 손실 감소 방향으로 갱신합니다.
    b -= lr * db  # 절편을 손실 감소 방향으로 갱신합니다.

test = np.array([3, 4, 5])  # 넘파이 배열을 만듭니다.
probs = sigmoid(w*test + b)  # 예측 확률을 저장합니다.
preds = (probs >= 0.5).astype(int)  # 자료형을 변환합니다.

print("확률:", probs)  # 결과를 화면에 출력합니다.
print("분류:", preds)  # 결과를 화면에 출력합니다.

```
</details>

In [ ]:
# 여기에 작성하세요

### 문제 4. 로지스틱 회귀 (scikit-learn)
사이킷런의 `LogisticRegression`을 사용해,  
X=[1,2,3,4,5], y=[0,0,0,1,1] 데이터를 학습하고,  
X=3, 4, 5의 예측 확률과 분류 결과를 출력하세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 모델입니다.
import numpy as np  # 배열 계산을 위한 Numpy입니다.

X = np.array([1, 2, 3, 4, 5]).reshape(-1, 1)  # 모델에 넣을 입력 데이터입니다.
y = np.array([0, 0, 0, 1, 1])  # 정답 값 또는 두 번째 배열을 준비합니다.

model = LogisticRegression()  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.
model.fit(X, y)  # 데이터로 모델이나 변환 기준을 학습합니다.

print("예측 확률:", model.predict_proba([[3], [4], [5]]))  # 각 클래스에 속할 확률을 예측합니다.
print("분류 결과:", model.predict([[3], [4], [5]]))  # 학습한 모델로 새 값을 예측합니다.

```
</details>



In [ ]:
# 여기에 작성하세요

### ✅ 체크포인트
- 선형 회귀는 연속적인 값을 예측, 로지스틱 회귀는 분류 문제에 사용된다.  
- 경사 하강법은 손실 함수를 줄이면서 파라미터를 학습하는 기본 알고리즘이다.  
- 정규방정식은 소규모 데이터에서 해를 빠르게 구할 수 있다.  
- `scikit-learn`을 사용하면 복잡한 수식을 직접 구현하지 않아도 쉽게 모델을 적용할 수 있다.  


#  머신러닝 더 빠르고 정확하게

머신러닝은 단순히 알고리즘만 아는 것으로 끝나지 않습니다.  
실제 현업에서는 **학습 속도가 느리거나**, **예측 성능이 기대보다 낮은 경우**가 자주 발생합니다.  

👉 이번 토픽은 모델 성능을 최적화하기 위한 핵심 기법들을 다룹니다:  
- 데이터 전처리  
- 규제(Regularization)  
- 모델 평가와 하이퍼파라미터 튜닝  

### 학습 목표

- 다양한 데이터 전처리 기법을 이해하고 적용할 수 있다.  
- 정규화 기법(L1, L2)을 이해하고 활용할 수 있다.  
- 교차 검증과 하이퍼파라미터 튜닝을 통해 모델 성능을 평가하고 개선할 수 있다.

### 목차

#### 1. 들어가기
- 왜 "빠르고 정확하게"가 중요한가?  
- 데이터 전처리 → 정규화 → 모델 평가 & 튜닝으로 이어지는 흐름  


#### 2. 데이터 전처리
- Feature Scaling
  - Normalization (0~1 범위)
  - Standardization (평균=0, 표준편차=1)
  - scikit-learn 실습
- One-hot Encoding
  - 범주형 데이터를 수치형으로 변환
  - pandas `get_dummies()` 실습



#### 3. 규제 (Regularization)
- Bias(편향) vs Variance(분산)  
- Bias-Variance Tradeoff 개념  
- 과적합 방지를 위한 규제 기법
  - L1 규제 (Lasso)
  - L2 규제 (Ridge)
- scikit-learn 실습: Lasso, Ridge 회귀 비교



#### 4. 모델 평가와 하이퍼파라미터 선택
- k겹 교차 검증 (k-Fold Cross Validation)  
  - scikit-learn `cross_val_score` 실습
- 그리드 서치 (Grid Search)  
  - scikit-learn `GridSearchCV` 실습
- 최적의 하이퍼파라미터 찾기  


### ✅ 체크포인트
- 전처리와 정규화는 모델 성능에 직접적으로 영향을 미친다.  
- Regularization은 과적합을 방지하는 핵심 도구이다.  
- 교차 검증과 그리드 서치는 모델 평가와 성능 개선의 표준 절차이다.


## 1. 들어가기

머신러닝 모델을 "더 빠르고 정확하게" 만들기 위해 가장 먼저 해야 할 일은 **데이터 전처리(Data Preprocessing)** 입니다.  
- 현실 데이터는 크기 단위가 제각각 (예: 키=cm, 몸무게=kg, 월수입=만원)  
- 숫자 범위가 다르면, 모델이 특정 특성에 **과도하게 영향을 받음**  
- 따라서 데이터를 적절히 스케일링(Scaling)하고 변환해야 학습이 잘 이루어집니다.


### 예시: 특성 간 범위 차이 문제

데이터에 두 개의 특성이 있다고 합시다:

- $x_1$: **키 비율** → 값의 범위: 0 ~ 1  
- $x_2$: **년 수입** → 값의 범위: 4000 ~ 10000  

### 문제점
- 두 특성을 그대로 사용하면, 모델은 계산 과정에서 **값의 크기가 큰 $x_2$ 1000~2000** 에 더 큰 가중치를 부여하게 됨.  
- 실제로는 $x_1$ (키 비율)도 중요한데, **숫자 스케일 차이 때문에 모델이 무시**할 수 있음.  


### 직관적 비유
- 어떤 학생의 성적을 예로 들어봅시다:
  - **과목 A (출석점수)**: 0~1점  
  - **과목 B (시험점수)**: 1000~2000점  

총점을 단순 합으로 계산하면?  
- 과목 A 점수는 아무리 변해도 1점 차이  
- 과목 B 점수는 최소 1000점 차이  

👉 당연히 **시험점수(과목 B)** 가 모든 결과를 좌우하게 됨 → 출석점수는 사실상 무시됨.  


### 해결 방법
- 데이터를 **스케일링**하여 두 특성이 비슷한 범위를 갖도록 변환해야 함.
- 예:
  - Min-Max Scaling → 모든 값을 0~1 사이로 맞춤  
  - Standardization → 평균=0, 표준편차=1로 변환  

이렇게 하면 모델이 **특성의 실제 중요도**를 제대로 반영할 수 있음.


## 2. 데이터 전처리

### 2.1 Feature Scaling (특성 스케일링)

#### (1) Normalization (정규화)
- 데이터 값을 **0~1 사이로 압축**  
- 공식:  
  $$
  x' = \frac{x - x_{min}}{x_{max} - x_{min}}
  $$


In [ ]:
import numpy as np  # 배열과 수치 연산 라이브러리입니다.
from sklearn.preprocessing import MinMaxScaler  # 0~1 정규화 변환기입니다.

data = np.array([[50], [200], [500]])  # 넘파이 배열을 만듭니다.
scaler = MinMaxScaler()  # 정규화 변환기입니다. 값을 0~1 범위로 맞춥니다.
normalized = scaler.fit_transform(data)  # 기준 학습과 변환을 한 번에 수행합니다.

print("원본 데이터:\n", data)  # 문자열을 정수로 바꿉니다.
print("정규화 데이터:\n", normalized)  # 문자열을 정수로 바꿉니다.

### (2) Standardization (표준화)
- 데이터의 평균=0, 표준편차=1로 맞춤  
- 공식:  
  
  $z = \frac{x - \mu}{\sigma}$

In [ ]:
from sklearn.preprocessing import StandardScaler  # 표준화 변환기입니다.

data = np.array([[50], [200], [500]])  # 넘파이 배열을 만듭니다.
scaler = StandardScaler()  # 표준화 변환기입니다. 평균 0, 표준편차 1로 맞춥니다.
standardized = scaler.fit_transform(data)  # 기준 학습과 변환을 한 번에 수행합니다.

print("표준화 데이터:\n", standardized)  # 문자열을 정수로 바꿉니다.

👉 참고 : 경사 하강법(Gradient Descent) 기반 모델은 **스케일링이 필수적**입니다.  
특성 범위가 다르면, 어떤 방향으로 먼저 학습할지 혼란이 생기기 때문입니다.  

### 2.2 One-hot Encoding (범주형 데이터 변환)

머신러닝 모델은 숫자만 처리할 수 있습니다.  
따라서 범주형 데이터(예: 성별=남/여, 지역=서울/부산/대구)는 **숫자 벡터**로 변환해야 합니다.  

- **문제점**: 단순히 "남=0, 여=1"처럼 하면, **순서/크기 관계가 생겨버림**  
- **해결책**: One-hot Encoding → 각 범주를 별도의 열로 분리, 해당 범주에만 1, 나머지는 0  

### 예시 (Gender 컬럼 변환 전/후)

| Index | Gender |
|-------|--------|
| 0     | Male   |
| 1     | Female |
| 2     | Female |
| 3     | Male   |

👇 One-hot Encoding 적용 후

| Index | Gender_Female | Gender_Male |
|-------|---------------|-------------|
| 0     | 0             | 1           |
| 1     | 1             | 0           |
| 2     | 1             | 0           |
| 3     | 0             | 1           |


In [ ]:
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.

df = pd.DataFrame({"Gender": ["Male", "Female", "Female", "Male"]})  # 데이터프레임을 직접 만듭니다.
encoded = pd.get_dummies(df, columns=["Gender"])  # 범주형 값을 원-핫 인코딩합니다.

print(df)  # 문자열을 정수로 바꿉니다.
print(encoded)  # 문자열을 정수로 바꿉니다.

👉 결과:  
- "Male" → [1, 0]  
- "Female" → [0, 1]  

### ✅ 체크포인트
- Normalization: 데이터 범위를 0~1 사이로 맞춤  
- Standardization: 평균=0, 표준편차=1로 맞춤  
- One-hot Encoding: 범주형 데이터를 숫자 벡터로 변환  
- Feature Scaling은 경사 하강법의 효율을 높이고, One-hot Encoding은 범주형 데이터 처리를 가능하게 한다.

## 3. 규제 (Regularization)

### 3.1 왜 규제가 필요한가?
머신러닝 모델은 훈련 데이터에 너무 **과적합(overfitting)** 되거나,  
너무 단순해서 **과소적합(underfitting)** 되는 경우가 많습니다.  

- **과소적합(Underfitting)**: 모델이 단순 → 데이터 패턴을 잘 못 잡음  
- **과적합(Overfitting)**: 모델이 복잡 → 훈련 데이터에는 잘 맞지만 새로운 데이터에서는 성능이 떨어짐  

  <img src="image/overfitting.png" width="500">

이미지 출처 : https://www.geeksforgeeks.org/machine-learning/underfitting-and-overfitting-in-machine-learning/

👉 규제(Regularization)는 **모델이 과적합되는 것을 막고, 일반화 성능을 높이는 방법**입니다.  


### 3.2 규제 개념
규제는 모델이 **불필요하게 큰 가중치**를 가지지 않도록 제약을 주어,  
과적합을 방지하고 일반화 성능을 높이는 방법입니다.


#### **L1 규제 (Lasso Regression)**  
- 가중치의 절댓값 합을 패널티로 부여  
- 일부 가중치를 0으로 만들어 **특성 선택(feature selection)** 효과  

$\text{Loss}_{L1} = \text{MSE} + \lambda \sum_i |w_i|$

예시:

- **규제 전:**  
$ y = 0.2x_1 + 235x_2 + 0.9x_3 $  
- **L1 규제 후:**  
$ y = 0x_1 + 1.3x_2 + 0x_3 $  
→ 일부 가중치가 **완전히 0**이 되어 불필요한 변수가 제거됨

#### **L2 규제 (Ridge Regression)**  
  - 가중치의 제곱합을 패널티로 부여
  - 모든 가중치를 조금씩 줄여 안정적인 모델 생성

  $\text{Loss}_{L2} = \text{MSE} + \lambda \sum_i w_i^2$

예시:

- **규제 전:**  
$y = 0.2x_1 + 235x_2 + 0.9x_3$ 
- **L2 규제 후:**  
$y = 0.1x_1 + 12.3x_2 + 0.5x_3$  
→ 모든 가중치가 **균등하게 작아짐** (0은 되지 않음)

####  **요약**

| 구분 | 패널티 항 | 효과 | 결과 |
|------|------------|--------|--------|
| **L1 규제 (Lasso)** | $\lambda \sum 절대값 w_i $ | 불필요한 변수 제거 | 일부 가중치 0 |
| **L2 규제 (Ridge)** | $\lambda \sum w_i^2$ | 가중치 크기 완화 | 모든 가중치 축소 |

In [ ]:
## 2.3 scikit-learn으로 과적합 문제 해결

from sklearn.linear_model import LinearRegression, Ridge, Lasso  # 선형 회귀 모델, L2 규제 회귀 모델입니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.metrics import mean_squared_error  # MSE 회귀 지표 함수입니다.
import numpy as np  # 배열과 수치 연산 라이브러리입니다.

# 샘플 데이터 생성
X = np.random.rand(100, 5) * 10  # 난수를 생성합니다.
y = 3*X[:,0] + 2*X[:,1] - X[:,2] + np.random.randn(100)*2  # 난수를 생성합니다.

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # 데이터를 훈련/평가용으로 나눕니다.

# 선형 회귀
lr = LinearRegression().fit(X_train, y_train)  # 선형 회귀 모델입니다.
print("LinearRegression MSE:", mean_squared_error(y_test, lr.predict(X_test)))  # 학습한 모델로 새 값을 예측합니다.

# Ridge 회귀 (L2)
ridge = Ridge(alpha=1.0).fit(X_train, y_train)  # 릿지 회귀입니다. alpha는 L2 규제 강도입니다.
print("Ridge MSE:", mean_squared_error(y_test, ridge.predict(X_test)))  # 학습한 모델로 새 값을 예측합니다.

# Lasso 회귀 (L1)
lasso = Lasso(alpha=0.1).fit(X_train, y_train)  # 라쏘 회귀입니다. alpha는 L1 규제 강도입니다.
print("Lasso MSE:", mean_squared_error(y_test, lasso.predict(X_test)))  # 학습한 모델로 새 값을 예측합니다.

### 3.4 L1, L2 직접 비교
- **L1 (Lasso)**: 일부 계수=0 → 불필요한 특성 제거 가능  
- **L2 (Ridge)**: 모든 계수를 작게 만들어 안정적인 모델  
- 실제로는 L1+L2 혼합한 **Elastic Net**도 많이 사용  


### ✅ 체크포인트
- 규제는 과적합 방지와 일반화 성능 향상에 핵심적이다.  
- L1(Lasso): 가중치 절댓값 합 → 특성 선택 효과  
- L2(Ridge): 가중치 제곱합 → 안정적인 모델  
- `alpha` 값이 클수록 규제 강도가 세지며, 너무 크면 과소적합 위험이 있다.  


## 4. 모델 평가와 하이퍼파라미터 선택

> 목적: **훈련 데이터에서만 잘 맞는 모델**을 피하고, **새로운 데이터에서도 일관되게 잘 작동(일반화)** 하도록 평가·튜닝한다.


### 4.1 왜 모델 평가가 중요한가?
- **훈련 성능 = 실제 성능 아님**: 훈련 데이터에 맞춘 점수는 낙관적일 수 있음(과적합).
- **일반화 확인**: 보지 못한 데이터(검증/테스트)에서 성능을 확인해야 함.
- **신뢰성**: 평가 절차가 재현 가능해야 하며, 데이터 누수(leakage)를 방지해야 함.


### 4.2 데이터 분할 전략

#### (1) Hold-out 분할(예: **8:1:1 = train:valid:test**)
- **train**: 학습
- **valid**: 하이퍼파라미터 선택/모델 비교
- **test**: 최종 성능 보고(딱 1번만 사용)


In [ ]:
from sklearn.datasets import load_iris  # 사이킷런 도구를 불러옵니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.

# 데이터 적재
# ------------------------------------------
iris = load_iris()
X, y = iris.data, iris.target

# ------------------------------------------
# 8:1:1 분할 (계층화 분할: 각 클래스 비율 유지)
# ------------------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(  # 데이터를 훈련/평가용으로 나눕니다.
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_valid, X_test, y_valid, y_test = train_test_split(  # 데이터를 훈련/평가용으로 나눕니다.
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print("train/valid/test:", X_train.shape, X_valid.shape, X_test.shape)  # 문자열을 정수로 바꿉니다.

#### 변형 예시: **6:2:2**, **7:1.5:1.5** 등. 데이터가 적으면 교차 검증을 권장.

### (2) 시계열(순서형) 데이터
#### 1. 특징
- 데이터는 시간 순서대로 기록됨 (예: 주식 가격, 날씨 데이터, 센서 데이터).
- **순서가 중요**하므로 무작위 섞기(shuffle) 금지.  
- 항상 **과거 → 미래** 순서를 지켜서 학습해야 함.  


#### 2. 올바른 분할 방식 (Sliding Window)
- 일반 데이터 분할(`train_test_split`)은 무작위 추출이 가능하지만, 시계열은 순서를 보존해야 함.  
- **Sliding Window**: 일정 길이의 과거 데이터를 묶어서(train window) 그 직후 미래 구간을 예측(valid window).  
  - 예: "과거 3일 → 다음 1일 예측"  
- 장점: 입력 시퀀스 길이가 일정 → 딥러닝/머신러닝 모델 학습에 바로 활용 가능.

#### 3. 코드 예시: Sliding Window 데이터셋 만들기

In [ ]:
import numpy as np  # 배열과 수치 연산 라이브러리입니다.

# 예제 시계열 데이터 (0 ~ 19)
series = np.arange(20)  # 범위 기반 숫자 배열을 만듭니다.

def make_window_data(series, window=5, horizon=1):
    X, y = [], []
    for i in range(len(series) - window - horizon + 1):
        X.append(series[i:i+window])            # 과거 구간
        y.append(series[i+window:i+window+horizon])  # 예측 구간
    return np.array(X), np.array(y)  # 넘파이 배열을 만듭니다.

# 과거 5일 데이터를 사용해 다음 1일을 예측
X, y = make_window_data(series, window=5, horizon=1)

print("X shape:", X.shape)  # (샘플 수, window 크기)
print("y shape:", y.shape)  # (샘플 수, horizon 크기)
print("첫 번째 샘플 X:", X[0], "-> y:", y[0])  # 문자열을 정수로 바꿉니다.

### 4.3 교차 검증(Cross Validation) 종류
- **KFold**: 무작위로 K분할 → 학습/검증 K회 반복 후 평균.  

- **StratifiedKFold**: 분류에서 **클래스 비율 유지**.
    - **예시**: 암 환자 데이터(환자 10%, 정상인 90%) → 일반 KFold는 어떤 fold엔 환자가 아예 없을 수도 있음.  
- **GroupKFold**: 동일 그룹은 같은 폴드에만 배치.
    - **예시**: 남/녀 를 맞춰야 할때 동일한 사람의 사진이 2장이상 존재할때 같은 그룹(train/val/test)에 배치
- **TimeSeriesSplit**: 시계열 전용(시간 순서 유지).


In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold  # 교차검증 점수 함수입니다.
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 분류 모델입니다.
from sklearn.datasets import load_iris  # 사이킷런 도구를 불러옵니다.
import numpy as np  # 배열과 수치 연산 라이브러리입니다.

iris = load_iris()

# 입력 특성 행렬(X)과 정답 벡터(y) 분리
# X : 꽃받침 길이, 꽃받침 폭, 꽃잎 길이, 꽃잎 폭 (4개의 특성)
# y : 붓꽃 품종 (0=setosa, 1=versicolor, 2=virginica)
X, y = iris.data, iris.target

# 로지스틱 회귀 모델 생성
# 분류(classification) 문제에 사용되는 선형 모델
# max_iter=200 : 학습 반복 횟수 제한 (기본값은 100, 수렴 문제 방지를 위해 늘림)
model = LogisticRegression(max_iter=200)  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.

# StratifiedKFold : 각 클래스 비율(레이블 분포)을 유지하면서 데이터셋을 여러 조각으로 나누는 K-겹 교차검증 방법
# n_splits=5 → 데이터를 5개의 폴드(fold)로 나눔 (즉, 5번의 학습/평가 수행)
# shuffle=True → 데이터를 무작위로 섞은 후 분할 (데이터 순서에 의한 편향 방지)
# random_state=42 → 난수 시드를 고정하여 재현성 확보
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")  # 교차검증 점수를 계산합니다.

# cross_val_score() : 교차 검증을 자동으로 수행해주는 함수
# 인자 설명:
#   model   : 사용할 학습 모델 (여기서는 로지스틱 회귀)
#   X, y    : 입력 데이터와 정답 레이블
#   cv      : 교차검증 분할 전략 (StratifiedKFold 객체)
#   scoring : 성능 평가 지표. 여기서는 'accuracy' (정확도)
#
# 실행 과정:
#   1. 데이터를 StratifiedKFold에 따라 5개의 폴드로 나눔
#   2. 각 폴드에 대해 1개는 테스트, 나머지 4개는 학습용으로 사용
#   3. 5번 반복하여 모델의 정확도를 계산
#   4. 각 반복에서의 정확도 점수를 배열 형태로 반환
print("교차 검증 점수:", scores)  # 문자열을 정수로 바꿉니다.
print("평균 정확도:", np.mean(scores))  # 평균을 계산합니다.

In [ ]:
# 각 폴드의 인덱스 확인
for fold_idx, (train_index, test_index) in enumerate(cv.split(X, y), start=1):
    print(f"\n📂 Fold {fold_idx}")  # 문자열을 정수로 바꿉니다.
    print(f"학습 데이터 인덱스 ({len(train_index)}개): {train_index[:10]} ...")  # 앞부분만 표시
    print(f"테스트 데이터 인덱스 ({len(test_index)}개): {test_index[:10]} ...")  # 문자열을 정수로 바꿉니다.
    print(f"→ 테스트 데이터의 클래스 분포: {np.bincount(y[test_index])}")  # 문자열을 정수로 바꿉니다.

#### **TIP**: 데이터 전처리(스케일링)는 반드시 **교차 검증 폴드 안에서** `fit`되어야 함 → `Pipeline` 사용!


### 4.4 평가 지표 선택 가이드


#### 1) 분류(Classification)

#### 혼동행렬(Confusion Matrix) 표

| 실제 \ 예측 | 0 (Negative)         | 1 (Positive)         |
|-------------|-----------------------|-----------------------|
| **0 (Negative)** | **TN**: 진짜 음성 (정상 정답) | **FP**: 거짓 양성 (거짓 경보) |
| **1 (Positive)** | **FN**: 거짓 음성 (놓침)     | **TP**: 진짜 양성 (정상 검출) |

#### 혼동행렬(Confusion Matrix) 용어 정리
- **TP (True Positive)**: 실제 1이고, 예측도 1  
- **FP (False Positive)**: 실제 0인데, 예측이 1 (거짓 경보)  
- **TN (True Negative)**: 실제 0이고, 예측도 0  
- **FN (False Negative)**: 실제 1인데, 예측이 0 (놓침)

#### **지표 공식**  
 - Accuracy = (TP + TN) / (TP + FP + TN + FN)  
 - Precision = TP / (TP + FP)  
 - Recall = TP / (TP + FN)  
 - F1 = 2 · (Precision · Recall) / (Precision + Recall)

| 지표 | 정의/설명 | 장점 | 주의할 점 / 활용 상황 |
|------|-----------|------|------------------------|
| **Accuracy** | 전체 샘플 중 정답 비율 | 직관적, 해석 쉬움 | 클래스 불균형(예: 정상 99%, 이상 1%)에 매우 취약 |
| **Precision (정밀도)** | `예측=양성` 중 실제 양성 비율 | 잘못된 경보(오탐) 줄이는 데 중요 | 양성 놓침(미탐)에는 둔감 |
| **Recall (재현율)** | 실제 양성 중 예측=양성 비율 | 양성 놓치지 않는 게 중요할 때 유리 | 오탐(거짓 양성)이 많아질 수 있음 |
| **F1 Score** | Precision·Recall의 조화 평균 | Precision·Recall 균형 평가 | 해석은 다소 어렵지만 불균형 데이터에 자주 사용 |
| **ROC-AUC** | 모든 임계값에서 TPR vs FPR 곡선 아래 면적 | 임계값에 독립적, 전반적 성능 평가 | 클래스 극심 불균형일 때 과대평가될 수 있음 |
| **PR-AUC** | Precision-Recall 곡선 아래 면적 | 양성 클래스 희소할 때 유리 | ROC-AUC보다 해석 어렵지만 불균형 심할 때 필수 |
| **Top-k Accuracy** | 다중 클래스에서 상위 k개 예측 안에 정답 있는지 | 이미지 분류(예: Top-5 Accuracy)에서 유리 | k 설정 필요 |

### 2) 회귀(Regression)

| 지표 | 정의/설명 | 장점 | 주의할 점 / 활용 상황 |
|------|-----------|------|------------------------|
| **MAE (Mean Absolute Error)** | 절댓값 오차 평균 | 해석 직관적, 이상치 영향 적음 | 큰 오차에 둔감 |
| **MSE (Mean Squared Error)** | 제곱 오차 평균 | 미분/최적화에 유리 | 큰 오차에 과도한 패널티 |
| **RMSE (Root Mean Squared Error)** | 제곱 오차의 제곱근 | 원 단위 복원, 큰 오차 강조 | MAE보다 이상치 민감 |
| **R² (결정계수)** | 모델이 데이터 분산을 얼마나 설명하는지 (1=완벽) | 상대적 성능 비교에 유용 | 데이터 분포에 따라 음수가 될 수 있음 |

In [ ]:
# scoring 예시: "accuracy", "f1_macro", "roc_auc_ovr", "neg_mean_absolute_error" 등

### 4.5 하이퍼파라미터란?
- **파라미터**: 모델이 **데이터로부터 학습**하는 값 (예: 모델의 가중치)
- **하이퍼파라미터**: 사람이 **사전에 설정**하는 값 (예: 정규화 강도 C/alpha, 트리 깊이, 학습률)

| 알고리즘 | 대표 하이퍼파라미터 | 의미/영향 |
|---|---|---|
| LogisticRegression | C, penalty, solver | 정규화 강도, 규제 형태 |
| SVM | C, kernel, gamma | 마진/복잡도 제어, 커널 폭 |
| RandomForest | n_estimators, max_depth, min_samples_split | 앙상블 크기/복잡도 |
| Ridge/Lasso | alpha | L2/L1 정규화 강도 |
| XGBoost/LightGBM | n_estimators, learning_rate, max_depth, subsample | 부스팅 단계/학습률/복잡도 |


### 4.6 그리드 서치(Grid Search) + 파이프라인(Pipeline) + 누수 방지

### 1) Grid Search란?
- 모델의 **하이퍼파라미터 후보 집합**을 정의하고,  
- 교차검증(Cross Validation)으로 모든 조합을 평가해 **최적 조합을 찾는 방법**.  

```python
from sklearn.model_selection import GridSearchCV  # 후보 조합 교차검증 도구입니다.
```


### 2) Pipeline이 필요한 이유

머신러닝 모델 학습에는 보통 이런 단계가 필요합니다:
```
[데이터] → [전처리기(예: 스케일러)] → [모델 학습]
```


#### ❌ 잘못된 방식 (누수 발생)
```python
from sklearn.preprocessing import StandardScaler  # 표준화 변환기입니다.
from sklearn.svm import SVC  # 사이킷런 도구를 불러옵니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)  # 데이터를 훈련/평가용으로 나눕니다.

scaler = StandardScaler()  # 표준화 변환기입니다. 평균 0, 표준편차 1로 맞춥니다.
scaler.fit(X)  # ⚠️ 전체 데이터로 평균/분산 학습 (X_train+X_test 모두 포함)

X_train_scaled = scaler.transform(X_train)  # 학습된 기준으로 데이터를 변환합니다.
X_test_scaled  = scaler.transform(X_test)  # 학습된 기준으로 데이터를 변환합니다.

model = SVC()
model.fit(X_train_scaled, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.
print(model.score(X_test_scaled, y_test))  # 모델의 기본 점수를 계산합니다.
```

- 여기서는 **테스트 데이터 정보까지 평균/분산 학습에 들어감**  
- 즉, 미래(테스트)의 정보를 미리 들여다본 셈 → 평가 점수가 실제보다 높게 나옴 (데이터 누수)


#### ✅ 올바른 방식 (Pipeline 사용)
```python
from sklearn.pipeline import Pipeline  # 전처리와 모델 연결 도구입니다.
from sklearn.svm import SVC  # 사이킷런 도구를 불러옵니다.
from sklearn.preprocessing import StandardScaler  # 표준화 변환기입니다.

pipe = Pipeline([  # 전처리와 모델을 한 흐름으로 묶습니다.
    ('scaler', StandardScaler()),  # step 1: 전처리
    ('svc', SVC())                 # step 2: 모델
])

pipe.fit(X_train, y_train)     # 내부적으로:
# scaler.fit(X_train) + scaler.transform(X_train)
# svc.fit(X_train_scaled, y_train)

pipe.predict(X_test)           # 내부적으로:
# scaler.transform(X_test) (train에서 학습한 평균/분산만 사용)
# svc.predict(X_test_scaled)
```

- `fit`을 호출하면 → train 데이터에서만 **전처리 fit**  
- `predict`를 호출하면 → train에서 학습한 기준(평균/분산)으로 test 데이터 변환 후 예측  
- 따라서 **test 데이터는 오직 평가용으로만 사용** → 누수 자동 방지


### ✅ 왜 Pipeline이 중요한가?
- 전처리와 모델을 따로 두면, 사용자가 실수로 `fit(X 전체)` 같은 코드를 짤 수 있음 → 누수 발생  
- **Pipeline은 전처리와 모델을 한 덩어리로 묶어서**,  
  - 학습(train 단계)에서는 train 데이터로만 전처리 fit  
  - 검증/테스트 단계에서는 transform만 적용  
- 즉, **사용자가 따로 구분해줄 필요 없이** 자동으로 안전하게 동작  


### 3) GridSearchCV + Pipeline 사용법

In [ ]:
from sklearn.datasets import load_iris  # 사이킷런 도구를 불러옵니다.
from sklearn.model_selection import GridSearchCV  # 후보 조합 교차검증 도구입니다.
from sklearn.pipeline import Pipeline  # 전처리와 모델 연결 도구입니다.
from sklearn.preprocessing import StandardScaler  # 표준화 변환기입니다.
from sklearn.svm import SVC  # 사이킷런 도구를 불러옵니다.

# 1. Pipeline 구성 (스텝에 이름 부여)
pipe = Pipeline([  # 전처리와 모델을 한 흐름으로 묶습니다.
    ('scaler', StandardScaler()),   #  step 1 전처리
    ('svc', SVC())                  #  step 2 모델
])

# 2. 탐색할 파라미터 (스텝이름__파라미터)
param_grid = {
    'svc__C': [0.1, 1, 10], # 'svc' 스텝의 C 파라미터
    'svc__kernel': ['linear', 'rbf'], # 'svc' 스텝의 kernel 파라미터
    'svc__gamma': ['scale', 'auto']
}

# 3. GridSearchCV 실행
grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy') # cv=5 : 5-fold 교차검증
grid.fit(*load_iris(return_X_y=True))  # 데이터로 모델이나 변환 기준을 학습합니다.

print("최적 파라미터:", grid.best_params_)  # 문자열을 정수로 바꿉니다.
print("최적 성능:", grid.best_score_)  # 문자열을 정수로 바꿉니다.

### 4) 핵심 정리
- **Pipeline**: 전처리 + 모델을 한 덩어리로 묶음.  
- **GridSearchCV**: 하이퍼파라미터를 전수조사해 최적값 선택.  
- **누수 방지**: 전처리는 fold별 train에서만 `fit`, valid/test는 `transform`만 적용.  
- **param_grid 키**: `스텝이름__파라미터` 형식으로 지정해야 함.  

### ✅ 요약 그림

```
[Raw Data] → [Scaler (fit on train only)] → [Model] → [Predict]
                ↑
                └─ Pipeline이 자동으로 누수 방지 보장
```


In [ ]:
from sklearn.pipeline import Pipeline  # 전처리와 모델 연결 도구입니다.
from sklearn.preprocessing import StandardScaler  # 표준화 변환기입니다.
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 분류 모델입니다.
from sklearn.model_selection import GridSearchCV, StratifiedKFold  # 후보 조합 교차검증 도구입니다.
from sklearn.metrics import accuracy_score, f1_score  # 정확도 지표 함수, F1 지표 함수입니다.

# ------------------------------------------
# 파이프라인: [스케일링 -> 로지스틱회귀]
# ------------------------------------------
pipe = Pipeline([  # 전처리와 모델을 한 흐름으로 묶습니다.
    ("scaler", StandardScaler()),  # 표준화 변환기입니다. 평균 0, 표준편차 1로 맞춥니다.
    ("clf", LogisticRegression(max_iter=200))  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.
])


# ------------------------------------------
# 하이퍼파라미터 그리드
#   - 파이프라인 스텝 이름 'clf__' 접두사로 접근
# ------------------------------------------
param_grid = {
    "clf__C": [0.01, 0.1, 1, 10, 100],
    "clf__penalty": ["l2"],            # liblinear/lbfgs 기준
    "clf__solver": ["liblinear", "lbfgs"]
}

# ------------------------------------------
# 교차 검증 설정(계층화 K-겹)
# ------------------------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ------------------------------------------
# GridSearchCV: scoring 변경 가능(불균형시 f1_macro 등 권장)
# ------------------------------------------
grid = GridSearchCV(  # 후보 파라미터를 교차검증으로 비교합니다.
    estimator=pipe,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.

print("최적 하이퍼파라미터:", grid.best_params_)  # 문자열을 정수로 바꿉니다.
print("검증 평균 점수(교차검증):", grid.best_score_)  # 문자열을 정수로 바꿉니다.

# ------------------------------------------
# 선택된 최적 모델로 검증/테스트 평가
# ------------------------------------------
best_model = grid.best_estimator_
valid_pred = best_model.predict(X_valid)  # 학습한 모델로 새 값을 예측합니다.
test_pred  = best_model.predict(X_test)  # 학습한 모델로 새 값을 예측합니다.

print("VALID accuracy:", accuracy_score(y_valid, valid_pred))  # 정확도 비율을 계산합니다.
print("TEST  accuracy:", accuracy_score(y_test,  test_pred))  # 정확도 비율을 계산합니다.

> **왜 Pipeline인가?**  
> 스케일러를 train+valid 전체에 미리 `fit`해버리면 **검증 정보가 훈련에 새는 데이터 누수** 발생.  
> Pipeline은 각 폴드마다 **훈련 파트에만 fit → 검증 파트에 transform**을 자동 적용한다.

### 연습문제. GridSearchCV의 학습 횟수 계산  

위 코드는 로지스틱 회귀(Logistic Regression)를 교차검증과 함께 수행하는 코드입니다.  
이때 모델 학습이 총 몇 번 일어나는지 계산하세요.  

 참고:  
- 교차검증: StratifiedKFold(n_splits=5)  
- 파라미터 조합: C(5개) × penalty(1개) × solver(2개)

총 학습 횟수를 구하세요.  


<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 필요한 값을 만들고 결과를 확인합니다.
# 문제에서 요구한 결과를 계산하고 확인합니다.

# 파라미터 조합 수 = 5 × 1 × 2 = 10
# 교차검증 fold 수 = 5
# 교차검증 중 학습 횟수 = 10 × 5 = 50
# 최적 조합으로 전체 훈련 세트 재학습 = 1회 추가

total_fits = 50 + 1  # 교차검증 50회와 최종 재학습 1회를 더합니다.
print("총 학습 횟수:", total_fits, "회")  # 계산한 총 학습 횟수를 출력합니다.

```
</details>


## 4.7 추가 팁 & 흔한 함정

- **Seed 고정**:  
  매번 실행할 때마다 결과가 달라지지 않게,  `random_state` 값을 고정하세요.  
  → 실험을 **다시 해도 같은 결과(재현성)** 가 나옵니다.

- **클래스 불균형**:  
  데이터에서 한 클래스(예: 0, 1)가 너무 많거나 적을 때는 `stratify` 옵션을 사용해 비율을 맞추고,  
  모델 학습 시 `class_weight='balanced'`를 주면 좋습니다. `-> 다수 클래스 예측시 패널티 부여`  
  → 평가 지표도 정확도(accuracy) 대신 **F1-score**나 **PR-AUC**가 더 적절합니다.

- **베이스라인**:  
  처음부터 복잡한 모델을 쓰기보다,  
  간단한 모델(예: 로지스틱 회귀, 선형 회귀)로 먼저 기준 점수를 만들어보세요.  
  → 이후 복잡한 모델(XGBoost, 딥러닝 등)이 정말 도움이 되는지 확인할 수 있습니다.

- **RandomizedSearchCV**:  
  하이퍼파라미터 후보가 너무 많을 때, 전부 다 시도(GridSearch)하는 대신 **일부만 랜덤하게 탐색**하는 게 효율적입니다.  
  → 빠르지만 충분히 좋은 조합을 찾을 수 있습니다.

- **특성 스케일링**:  
  SVM, KNN, 로지스틱 회귀처럼 **거리나 크기를 계산하는 모델**에서는 각 특성(컬럼)의 단위를 맞춰주는 게 중요합니다.  
  → `StandardScaler`(평균 0, 표준편차 1)나 `MinMaxScaler`(0~1 정규화)를 꼭 사용하세요.

- **데이터 누수(Data Leakage)**:  
  스케일링, 인코딩, 특성 선택 등은  **훈련 데이터에만** 맞춰서 해야 합니다.  
  → 그래서 이런 전처리는 항상 `Pipeline` 안에 넣어  
    교차검증 때마다 자동으로 훈련 세트 기준으로 적용되게 해야 합니다.


### ✅ 체크포인트
- [ ] 8:1:1 분할 또는 교차 검증으로 **일반화 성능**을 확인했는가?  
- [ ] 전처리·모델을 **Pipeline**으로 묶어 **누수**를 방지했는가?  
- [ ] 문제 특성에 맞는 **평가지표**를 선택했는가?  
- [ ] 합리적인 **하이퍼파라미터 공간**을 정의했는가? 

## 머신러닝 더 빠르고 정확하게 – 실습 문제

In [ ]:
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.
titanic = pd.read_csv("titanic/train.csv")  # CSV 파일을 데이터프레임으로 읽습니다.

### 문제 1. Feature Scaling
Titanic 데이터셋에서 `Age` 컬럼을 불러와서 **정규화(Normalization)** 와 **표준화(Standardization)** 를 적용해보세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 필요한 값을 만들고 결과를 확인합니다.
# 문제에서 요구한 결과를 계산하고 확인합니다.

import pandas as pd  # 표 데이터를 다루기 위한 Pandas입니다.
from sklearn.preprocessing import MinMaxScaler, StandardScaler  # 표준화 변환기, 0~1 정규화 변환기입니다.

# 데이터 불러오기
titanic = pd.read_csv("titanic/train.csv")  # Titanic 데이터를 DataFrame으로 불러옵니다.

age = titanic[["Age"]].dropna()# 결측치 제거

# 정규화
minmax = MinMaxScaler().fit_transform(age)  # 학습과 변환을 한 번에 수행합니다.
print("정규화 결과:\n", minmax[:5])  # 결과를 화면에 출력합니다.

# 표준화
standard = StandardScaler().fit_transform(age)  # 학습과 변환을 한 번에 수행합니다.
print("표준화 결과:\n", standard[:5])  # 결과를 화면에 출력합니다.

```
</details>

In [ ]:
# 여기에 정답을 작성하세요


### 문제 2. One-hot Encoding
Titanic 데이터셋의 `Sex` 컬럼을 One-hot Encoding하여 새로운 DataFrame을 만들어보세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 필요한 값을 만들고 결과를 확인합니다.
# 문제에서 요구한 결과를 계산하고 확인합니다.

encoded = pd.get_dummies(titanic[["Sex"]], prefix="Sex")  # 범주형 값을 원-핫 인코딩합니다.
print(encoded.head())  # 결과를 화면에 출력합니다.

```
</details>

In [ ]:
# 여기에 정답을 작성하세요


### 문제 3. L1 / L2 정규화
Titanic 데이터셋에서 `Pclass`, `Survived`, `Fare`를 입력 변수로, `Age`를 타깃 변수로 하여  
Lasso(L1)와 Ridge(L2) 회귀를 각각 적용해보고 성능(결정계수)을 비교하세요.  

결정계수 R²란?
- **결정계수(R-squared, R²)** 는 회귀모델이 **데이터를 얼마나 잘 설명하는지** 나타내는 지표입니다.  
- 예측값이 실제값에 얼마나 가까운지를 평가하며, 0~1 사이의 값(또는 그 이하)로 나타납니다.  

| R² 값 | 의미 |
|-------|------|
| **1.0** | 완벽하게 예측 (모든 점이 회귀선 위에 있음) |
| **0.0** | 평균값으로만 예측하는 것과 동일한 수준 |
| **0 미만** | 모델이 평균으로 예측하는 것보다 더 나쁨 (잘못 학습된 경우) |

✅ **해석 방법**  
- R²가 높을수록 모델이 데이터를 잘 설명한다는 의미입니다.  
- 단, R²가 높다고 항상 좋은 모델은 아니며, **과적합(overfitting)** 여부도 함께 확인해야 합니다.



<details>
<summary>정답 보기</summary>

```python 
from sklearn.linear_model import Lasso, Ridge  # L2 규제 회귀 모델, L1 규제 회귀 모델입니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.metrics import accuracy_score  # 정확도 지표 함수입니다.
import numpy as np  # 배열과 수치 연산 라이브러리입니다.

# 간단한 전처리
df = titanic[["Survived", "Pclass",  "Survived", "Age", "Fare"]].dropna()  # 결측치가 있는 행/열을 제거합니다.
X = df[["Pclass", "Survived", "Fare"]]
y = df["Age"]

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # 데이터를 훈련/평가용으로 나눕니다.

# Ridge (L2)
ridge = Ridge(alpha=1.0)  # 릿지 회귀입니다. alpha는 L2 규제 강도입니다.
ridge.fit(X_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.
print("Ridge score:", ridge.score(X_test, y_test)) # .score : R² (결정계수) 

# R² = 1 → 완벽하게 예측
# R² = 0 → 평균값으로 예측하는 것과 동일한 수준
# R² < 0 → 모델이 평균으로 예측하는 것보다 오히려 못함

# Lasso (L1)
lasso = Lasso(alpha=0.01)  # 라쏘 회귀입니다. alpha는 L1 규제 강도입니다.
lasso.fit(X_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.
print("Lasso score:", lasso.score(X_test, y_test))  # 모델의 기본 점수를 계산합니다.
```
</details>



In [ ]:
# 여기에 정답을 작성하세요


### 문제 4. k겹 교차 검증
Titanic 데이터셋에서 `Pclass`, `Sex`, `Age`를 사용해 `Survived`를 예측하는 로지스틱 회귀를 학습하고, 5겹 교차 검증으로 평균 정확도를 구하세요.  

<details>
<summary>정답 보기</summary>

```python 
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 분류 모델입니다.
from sklearn.model_selection import cross_val_score  # 교차검증 점수 함수입니다.
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.

titanic = pd.read_csv('titanic/train.csv')  # CSV 파일을 데이터프레임으로 읽습니다.

# 간단한 전처리
df = titanic[["Survived", "Pclass", "Sex", "Age"]].dropna()  # 결측치가 있는 행/열을 제거합니다.
df = pd.get_dummies(df, columns=["Sex"])  # 범주형 값을 원-핫 인코딩합니다.
X = df.drop("Survived", axis=1)  # 불필요한 열이나 행을 제거합니다.
y = df["Survived"]

model = LogisticRegression(max_iter=200)  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.
scores = cross_val_score(model, X, y, cv=5)  # 교차검증 점수를 계산합니다.

print("교차 검증 점수:", scores)  # 문자열을 정수로 바꿉니다.
print("평균 정확도:", scores.mean())  # 문자열을 정수로 바꿉니다.
```
</details>



In [ ]:
# 여기에 정답을 작성하세요
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 분류 모델입니다.
from sklearn.model_selection import cross_val_score  # 교차검증 점수 함수입니다.
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.

titanic = pd.read_csv('titanic/train.csv')  # CSV 파일을 데이터프레임으로 읽습니다.

### 문제 5. 그리드 서치
Titanic 데이터셋에서 로지스틱 회귀 모델을 사용하고, `C` 값 후보 [0.01, 0.1, 1, 10]에 대해 GridSearchCV로 최적 하이퍼파라미터를 찾아보세요.  

<details>
<summary>정답 보기</summary>

```python 
from sklearn.model_selection import GridSearchCV  # 후보 조합 교차검증 도구입니다.

titanic = pd.read_csv('titanic/train.csv')  # CSV 파일을 데이터프레임으로 읽습니다.
df = titanic[["Survived", "Pclass", "Sex", "Age"]].dropna()  # 결측치가 있는 행/열을 제거합니다.
df = pd.get_dummies(df, columns=["Sex"])  # 범주형 값을 원-핫 인코딩합니다.
X = df.drop("Survived", axis=1)  # 불필요한 열이나 행을 제거합니다.
y = df["Survived"]

param_grid = {"C": [0.01, 0.1, 1, 10]}
grid = GridSearchCV(LogisticRegression(max_iter=200), param_grid, cv=5)  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.
grid.fit(X, y)  # 데이터로 모델이나 변환 기준을 학습합니다.

print("최적 파라미터:", grid.best_params_)  # 문자열을 정수로 바꿉니다.
print("최적 성능:", grid.best_score_)  # 문자열을 정수로 바꿉니다.
```
</details>

In [ ]:
# 여기에 정답을 작성하세요
from sklearn.model_selection import GridSearchCV  # 후보 조합 교차검증 도구입니다.

titanic = pd.read_csv('titanic/train.csv')  # CSV 파일을 데이터프레임으로 읽습니다.

### 문제 6. 데이터 분할 (Train/Validation/Test)
Titanic 데이터셋을 **8:1:1 비율**로 학습, 검증, 테스트 세트로 나누어보세요.  

<details>
<summary>정답 보기</summary>

```python 
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.

df = titanic[["Survived", "Pclass", "Age", "Fare"]].dropna()  # 결측치가 있는 행/열을 제거합니다.
X = df.drop("Survived", axis=1)  # 불필요한 열이나 행을 제거합니다.
y = df["Survived"]

# 8:2 분할 → 0.2 중 절반은 검증, 절반은 테스트
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)  # 데이터를 훈련/평가용으로 나눕니다.
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)  # 데이터를 훈련/평가용으로 나눕니다.

print(X_train.shape, X_valid.shape, X_test.shape)  # 문자열을 정수로 바꿉니다.
```
</details>



In [ ]:
# 여기에 정답을 작성하세요
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.

titanic = pd.read_csv('titanic/train.csv')  # CSV 파일을 데이터프레임으로 읽습니다.

### 문제 7. 표준화 + 로지스틱 회귀
Titanic 데이터셋에서 `Age`, `Fare`를 입력 변수로 사용하고, **표준화(StandardScaler)** 를 적용한 후 로지스틱 회귀를 학습하세요.  

<details>
<summary>정답 보기</summary>

```python 
from sklearn.preprocessing import StandardScaler  # 표준화 변환기입니다.
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 분류 모델입니다.

df = titanic[["Survived", "Age", "Fare"]].dropna()  # 결측치가 있는 행/열을 제거합니다.
X = df[["Age", "Fare"]]
y = df["Survived"]

scaler = StandardScaler()  # 표준화 변환기입니다. 평균 0, 표준편차 1로 맞춥니다.
X_scaled = scaler.fit_transform(X)  # 기준 학습과 변환을 한 번에 수행합니다.

model = LogisticRegression(max_iter=200)  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.
model.fit(X_scaled, y)  # 데이터로 모델이나 변환 기준을 학습합니다.

print("모델 정확도:", model.score(X_scaled, y))  # 모델의 기본 점수를 계산합니다.

</details>
```


In [ ]:
# 여기에 정답을 작성하세요
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.

titanic = pd.read_csv('titanic/train.csv')  # CSV 파일을 데이터프레임으로 읽습니다.

### 문제 8. ROC-AUC 평가
Titanic 데이터셋에서 `Pclass`, `Sex`, `Age`를 사용해 로지스틱 회귀를 학습하고, **ROC-AUC 점수**를 계산하세요.  

ROC-AUC란?
- 분류 모델의 **예측 확률 품질**을 평가하는 지표입니다.  
- 단순 정확도(accuracy)는 “맞았다/틀렸다”만 보지만,  
  **ROC-AUC는 모델이 긍정 클래스(예: 생존)를 얼마나 잘 구분하는지**를 확률 기반으로 평가합니다.  

  <img src="image/rocauc.webp" width="500">

ROC (Receiver Operating Characteristic) 곡선
- 모델이 다양한 **분류 기준(threshold)** 에서  
  **True Positive Rate(재현율, TPR)** 과  
  **False Positive Rate(거짓 양성률, FPR, FP / (FP + TN))** 의 관계를 나타낸 곡선입니다.  
- 쉽게 말해, **민감도와 오탐률의 트레이드오프 곡선**입니다.  


AUC (Area Under the Curve)
- ROC 곡선 아래의 면적(Area)을 계산한 값입니다.  
- 값의 범위는 0~1이며,  
  **1에 가까울수록 모델이 잘 구분한다는 의미**입니다.

| AUC 값 | 모델 성능 해석 |
|---------|----------------|
| 1.0 | 완벽한 분류 |
| 0.9 이상 | 매우 우수 |
| 0.8 이상 | 양호 |
| 0.5 | 랜덤 추측 수준 |
| 0.5 미만 | 오히려 반대로 예측 |  
  
   
<details>
<summary>정답 보기</summary>

```python 
from sklearn.metrics import roc_auc_score  # 사이킷런 도구를 불러옵니다.

df = titanic[["Survived", "Pclass", "Sex", "Age"]].dropna()  # 결측치가 있는 행/열을 제거합니다.
df = pd.get_dummies(df, columns=["Sex"])  # 범주형 값을 원-핫 인코딩합니다.
X = df.drop("Survived", axis=1)  # 불필요한 열이나 행을 제거합니다.
y = df["Survived"]

model = LogisticRegression(max_iter=200)  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.
model.fit(X, y)  # 데이터로 모델이나 변환 기준을 학습합니다.
probs = model.predict_proba(X)[:, 1]  # 클래스별 예측 확률을 계산합니다.

print("ROC-AUC:", roc_auc_score(y, probs))  # 문자열을 정수로 바꿉니다.
```
</details>



In [ ]:
# 여기에 정답을 작성하세요
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.metrics import roc_auc_score  # 사이킷런 도구를 불러옵니다.

titanic = pd.read_csv('titanic/train.csv')  # CSV 파일을 데이터프레임으로 읽습니다.

### 문제 9. 교차 검증 지표 변경
Titanic 데이터셋에서 로지스틱 회귀 모델을 학습하고, **교차 검증 시 Accuracy가 아닌 F1-score**를 사용해 평균 성능을 구하세요.  

<details>
<summary>정답 보기</summary>

```python 
scores = cross_val_score(LogisticRegression(max_iter=200), X, y, cv=5, scoring="f1")  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.
print("평균 F1-score:", scores.mean())  # 문자열을 정수로 바꿉니다.
```
</details>



In [ ]:
# 여기에 정답을 작성하세요
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.

titanic = pd.read_csv('titanic/train.csv')  # CSV 파일을 데이터프레임으로 읽습니다.


### 문제 10. 파이프라인(Pipeline) 활용
Titanic 데이터셋에서 `Age`, `Pclass`, `Sex`를 입력 변수로 사용하여,  
**[표준화 → 로지스틱 회귀]** 단계를 Pipeline으로 구성하고 정확도를 계산하세요.  

<details>
<summary>정답 보기</summary>

```python 
from sklearn.pipeline import Pipeline  # 전처리와 모델 연결 도구입니다.

# 간단한 전처리
df = titanic[["Survived", "Pclass", "Sex", "Age"]].dropna()  # 결측치가 있는 행/열을 제거합니다.
df = pd.get_dummies(df, columns=["Sex"])  # 범주형 값을 원-핫 인코딩합니다.
X = df.drop("Survived", axis=1)  # 불필요한 열이나 행을 제거합니다.
y = df["Survived"]

pipe = Pipeline([  # 전처리와 모델을 한 흐름으로 묶습니다.
    ("scaler", StandardScaler()),  # 표준화 변환기입니다. 평균 0, 표준편차 1로 맞춥니다.
    ("clf", LogisticRegression(max_iter=200))  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.
])

pipe.fit(X, y)  # 데이터로 모델이나 변환 기준을 학습합니다.
print("정확도:", pipe.score(X, y))  # 모델의 기본 점수를 계산합니다.
```
</details>



In [ ]:
# 여기에 정답을 작성하세요
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.pipeline import Pipeline  # 전처리와 모델 연결 도구입니다.

titanic = pd.read_csv('titanic/train.csv')  # CSV 파일을 데이터프레임으로 읽습니다.

# 간단한 전처리
df = titanic[["Survived", "Pclass", "Sex", "Age"]].dropna()  # 결측치가 있는 행/열을 제거합니다.
df = pd.get_dummies(df, columns=["Sex"])  # 범주형 값을 원-핫 인코딩합니다.
X = df.drop("Survived", axis=1)  # 불필요한 열이나 행을 제거합니다.
y = df["Survived"]

### 문제 11. Randomized Search
Titanic 데이터셋에서 로지스틱 회귀의 `C` 값을 [0.001, 0.01, 0.1, 1, 10, 100] 중에서,  
**RandomizedSearchCV**를 사용해 최적 하이퍼파라미터를 찾아보세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 필요한 값을 만들고 결과를 확인합니다.
# 문제에서 요구한 결과를 계산하고 확인합니다.

from sklearn.model_selection import RandomizedSearchCV  # 무작위 후보 탐색 도구입니다.
1
param_dist = {"C": [0.001, 0.01, 0.1, 1, 10, 100]}  # 탐색할 하이퍼파라미터 후보입니다.
search = RandomizedSearchCV(LogisticRegression(max_iter=200), param_dist, cv=5, n_iter=3, random_state=42)  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.
search.fit(X, y)  # 데이터로 모델이나 변환 기준을 학습합니다.

print("최적 파라미터:", search.best_params_)  # 결과를 화면에 출력합니다.
print("최적 점수:", search.best_score_)  # 결과를 화면에 출력합니다.

```
</details>



In [ ]:
# 여기에 정답을 작성하세요
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.model_selection import RandomizedSearchCV  # 무작위 후보 탐색 도구입니다.
from sklearn.pipeline import Pipeline  # 전처리와 모델 연결 도구입니다.

titanic = pd.read_csv('titanic/train.csv')  # CSV 파일을 데이터프레임으로 읽습니다.

# 간단한 전처리
df = titanic[["Survived", "Pclass", "Sex", "Age"]].dropna()  # 결측치가 있는 행/열을 제거합니다.
df = pd.get_dummies(df, columns=["Sex"])  # 범주형 값을 원-핫 인코딩합니다.
X = df.drop("Survived", axis=1)  # 불필요한 열이나 행을 제거합니다.
y = df["Survived"]

### 문제 12. 특성 중요도 확인 (Lasso)
Titanic 데이터셋에서 Lasso(L1) 회귀를 학습한 후 **계수(coefficient)** 값을 출력해보세요.  
참고 : model.coef_

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 필요한 값을 만들고 결과를 확인합니다.
# 문제에서 요구한 결과를 계산하고 확인합니다.

lasso = Lasso(alpha=0.01)  # 라쏘 회귀입니다. alpha는 L1 규제 강도입니다.
lasso.fit(X, y)  # 데이터로 모델이나 변환 기준을 학습합니다.

print("계수:", lasso.coef_)  # 결과를 화면에 출력합니다.
# 계수: [-0.19104762 -0.00527471  0.43988593 -0.        ]
# x₁이 1 증가하면 y가 약 0.19 감소
# x₂가 1 증가하면 y가 거의 변하지 않음 (영향 미미)
# x₃이 1 증가하면 y가 약 0.44 증가
# L1 규제에 의해 0으로 수축 → 이 변수는 영향이 없다고 판단됨

```
</details>



In [ ]:
# 여기에 정답을 작성하세요
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.pipeline import Pipeline  # 전처리와 모델 연결 도구입니다.

titanic = pd.read_csv('titanic/train.csv')  # CSV 파일을 데이터프레임으로 읽습니다.

# 간단한 전처리
df = titanic[["Survived", "Pclass", "Sex", "Age"]].dropna()  # 결측치가 있는 행/열을 제거합니다.
df = pd.get_dummies(df, columns=["Sex"])  # 범주형 값을 원-핫 인코딩합니다.
X = df.drop("Survived", axis=1)  # 불필요한 열이나 행을 제거합니다.
y = df["Survived"]

### 문제 13. Confusion Matrix
Titanic 데이터셋에서 로지스틱 회귀 모델을 학습하고,  
**혼동 행렬(confusion matrix)** 을 출력해보세요.  

<details>
<summary>정답 보기</summary>

```python 
from sklearn.metrics import confusion_matrix  # 혼동행렬 함수입니다.

df = titanic[["Survived", "Pclass", "Age", "Fare"]].dropna()  # 결측치가 있는 행/열을 제거합니다.
X = df.drop("Survived", axis=1)  # 불필요한 열이나 행을 제거합니다.
y = df["Survived"]

model = LogisticRegression(max_iter=200)  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.
model.fit(X, y)  # 데이터로 모델이나 변환 기준을 학습합니다.
y_pred = model.predict(X)  # 학습한 모델로 새 값을 예측합니다.

print(confusion_matrix(y, y_pred))  # 실제값과 예측값의 혼동행렬입니다.
```
</details>



In [ ]:
# 여기에 정답을 작성하세요
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.metrics import confusion_matrix  # 혼동행렬 함수입니다.
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 분류 모델입니다.

df = titanic[["Survived", "Pclass", "Age", "Fare"]].dropna()  # 결측치가 있는 행/열을 제거합니다.
X = df.drop("Survived", axis=1)  # 불필요한 열이나 행을 제거합니다.
y = df["Survived"]

### 문제 14. Classification Report
Titanic 데이터셋에서 로지스틱 회귀 모델을 학습하고,  
**Precision, Recall, F1-score**를 포함한 Classification Report를 출력하세요.  

<details>
<summary>정답 보기</summary>

```python  
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.metrics import classification_report  # 분류 지표 요약 함수입니다.
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 분류 모델입니다.

df = titanic[["Survived", "Pclass", "Age", "Fare"]].dropna()  # 결측치가 있는 행/열을 제거합니다.
X = df.drop("Survived", axis=1)  # 불필요한 열이나 행을 제거합니다.
y = df["Survived"]

# 8:2 분할 → 0.2 중 절반은 검증, 절반은 테스트
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)  # 데이터를 훈련/평가용으로 나눕니다.
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)  # 데이터를 훈련/평가용으로 나눕니다.

model = LogisticRegression()  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.

model.fit(X_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.

y_pred = model.predict(X_test)  # 학습한 모델로 새 값을 예측합니다.

print(classification_report(y_test, y_pred))  # 정밀도/재현율/F1을 요약합니다.
```
</details>

In [ ]:
# 여기에 정답을 작성하세요

from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.metrics import classification_report  # 분류 지표 요약 함수입니다.
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 분류 모델입니다.

df = titanic[["Survived", "Pclass", "Age", "Fare"]].dropna()  # 결측치가 있는 행/열을 제거합니다.
X = df.drop("Survived", axis=1)  # 불필요한 열이나 행을 제거합니다.
y = df["Survived"]

### ✅ 체크포인트
- 전처리(스케일링, 원-핫 인코딩)는 모델 학습에 필수적이다.  
- 정규화(L1, L2)는 과적합을 방지하고 일반화 성능을 높인다.  
- k겹 교차 검증은 모델 안정성을 평가하는 좋은 방법이다.  
- Grid Search는 최적 하이퍼파라미터를 자동으로 찾아준다.  

## 캐글 타이타닉 : https://www.kaggle.com/competitions/titanic/overview
### 목표 : 전처리 방법 변경 및 모델을 Tensorflow 딥러닝 모델로 변경하여 제출 후 스코어 0.8 이상 도달하기

### baseline

In [ ]:
# 필요한 라이브러리 임포트
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.
import numpy as np  # 배열과 수치 연산 라이브러리입니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.ensemble import RandomForestClassifier  # 랜덤포레스트 분류 모델입니다.
from sklearn.metrics import accuracy_score, classification_report  # 정확도 지표 함수, 분류 지표 요약 함수입니다.

# 데이터 불러오기
train = pd.read_csv('../titanic/train.csv')  # CSV 파일을 데이터프레임으로 읽습니다.
test = pd.read_csv('../titanic/test.csv')  # CSV 파일을 데이터프레임으로 읽습니다.

# 데이터 전처리
def preprocess_data(df):
    # 결측치 처리
    df['Age'] = df['Age'].fillna(df['Age'].mean())  # 결측치를 지정한 값으로 채웁니다.
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])  # 결측치를 지정한 값으로 채웁니다.
    df['Fare'] = df['Fare'].fillna(df['Fare'].mean())  # 결측치를 지정한 값으로 채웁니다.
    
    # 범주형 변수 처리
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})  # 값을 지정한 규칙으로 바꿉니다.
    df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})  # 값을 지정한 규칙으로 바꿉니다.
    
    # 필요한 특성 선택
    features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
    return df[features]

# 학습 데이터 전처리
X = preprocess_data(train)
y = train['Survived']

# 학습 데이터와 검증 데이터 분리
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)  # 데이터를 훈련/평가용으로 나눕니다.

# RandomForest 모델 생성 및 학습
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)  # 랜덤포레스트입니다. n_estimators는 트리 수입니다.
rf_model.fit(X_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.

# 검증 데이터로 예측
val_pred = rf_model.predict(X_val)  # 학습한 모델로 새 값을 예측합니다.

# 모델 성능 평가
print('검증 데이터 정확도:', accuracy_score(y_val, val_pred))  # 정확도 비율을 계산합니다.
print('\n분류 보고서:')  # 문자열을 정수로 바꿉니다.
print(classification_report(y_val, val_pred))  # 정밀도/재현율/F1을 요약합니다.

# 테스트 데이터 예측
test_processed = preprocess_data(test)
test_pred = rf_model.predict(test_processed)  # 학습한 모델로 새 값을 예측합니다.

# 제출 파일 생성
submission = pd.DataFrame({  # 데이터프레임을 직접 만듭니다.
    'PassengerId': test['PassengerId'],
    'Survived': test_pred
})
submission.to_csv('titanic_submission.csv', index=False)
print('\n제출 파일이 생성되었습니다.')  # 문자열을 정수로 바꿉니다.
